In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

#Init

In [0]:
from bronze_config import INGESTION_CONFIG
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampNTZType,DecimalType, CharType,ByteType, DoubleType, LongType
from pyspark.sql.functions import col, lit
from IPython.display import display
from pyspark.sql import functions as F
import numpy as np
import pandas as pd

In [0]:
%sql
USE CATALOG nyc_taxi_project;
USE SCHEMA bronze;

# Read from files an Write Bronze Tables

In [0]:
taxi_schema=StructType(
    [
        StructField("VendorID", IntegerType(), True),
        StructField("tpep_pickup_datetime", TimestampNTZType(), True),
        StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
        StructField("passenger_count", LongType(), True),
        StructField("trip_distance",DoubleType(),True),
        StructField("RatecodeID", LongType(), True),
        StructField("store_and_fwd_flag",StringType(), True),
        StructField("PULocationID", IntegerType(), True),
        StructField("DOLocationID", IntegerType(), True),
        StructField("payment_type", LongType(), True),
        StructField("fare_amount", DoubleType(), True),
        StructField("extra", DoubleType(), True),
        StructField("mta_tax", DoubleType(), True),
        StructField("tip_amount", DoubleType(), True),
        StructField("tolls_amount", DoubleType(), True),
        StructField("improvement_surcharge", DoubleType(), True),
        StructField("total_amount", DoubleType(), True),
        StructField("congestion_surcharge", DoubleType(), True),
        StructField("Airport_fee", DoubleType(), True),
        StructField("cbd_congestion_fee", DoubleType(), True)

    ]
)

In [0]:
for item in INGESTION_CONFIG:
    print(f"Ingestion {item['source']} -> bronze.{item['table']}")
    if item['file_type'] == "csv":
        df= (spark.read.format(item['file_type']).option("header", "true").option("inferSchema", "true").load(item['path']))
    elif item['file_type'] == "parquet":
        df= (spark.read.format(item['file_type']).schema(taxi_schema).load(item['path']))
      
    (df.write.mode("overwrite").format("delta").saveAsTable(f"bronze.{item['table']}"))
    print(f"{item['path']} successfully loaded to bronze.{item['table']}")

In [0]:
%sql
select * from bronze.tlc_taxi_lkp

In [0]:
%sql
select * from bronze.yellow_tripdata_202501